In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import uuid

# ============================================================
# 1. Source / Target
# ============================================================

source_path = (
    "/Volumes/customer360_dev/staging/"
    "source_files/addresses/incremental/"
)

target_table = "customer360_dev.bronze.addresses"

run_id = str(uuid.uuid4())

print("Pipeline Run ID:", run_id)


# ============================================================
# 2. Explicit Address Schema
# ============================================================

address_schema = StructType([
    StructField("address_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("address_type", StringType(), True),
    StructField("address_line1", StringType(), True),
    StructField("address_line2", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("is_primary", StringType(), True),
    StructField("effective_from", StringType(), True),
    StructField("effective_to", StringType(), True),
    StructField("is_current", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True)
])


# ============================================================
# 3. Read Incremental CSV Files
# ============================================================

incremental_df = (
    spark.read
    .option("header", True)
    .schema(address_schema)
    .csv(source_path)
)


# ============================================================
# 4. Add Bronze Metadata
# ============================================================

bronze_incremental_df = (
    incremental_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_system",
        F.lit("CRM")
    )
    .withColumn(
        "_load_type",
        F.lit("INCREMENTAL")
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(run_id)
    )
)


# ============================================================
# 5. Preview
# ============================================================

display(
    bronze_incremental_df.limit(20)
)


# ============================================================
# 6. Count Incoming Rows
# ============================================================

incoming_count = bronze_incremental_df.count()

print(
    f"Incoming address incremental rows: "
    f"{incoming_count:,}"
)


# ============================================================
# 7. Append to Bronze
# ============================================================

if incoming_count > 0:

    (
        bronze_incremental_df
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(
            target_table
        )
    )

    print(
        f"Address incremental batch loaded. "
        f"run_id = {run_id}"
    )

else:

    print(
        "No incremental address records found. "
        "Bronze write skipped."
    )

Pipeline Run ID: 679af461-284d-4c49-830f-078390990f99


address_id,customer_id,address_type,address_line1,address_line2,city,state,postal_code,country,is_primary,effective_from,effective_to,is_current,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,_pipeline_run_id


Incoming address incremental rows: 0
No incremental address records found. Bronze write skipped.


In [0]:
if incoming_count > 0:

    written_count = (
        spark.table(target_table)
        .filter(
            F.col("_pipeline_run_id") == run_id
        )
        .count()
    )

else:

    written_count = 0

print(f"Incoming rows : {incoming_count:,}")
print(f"Written rows  : {written_count:,}")

Incoming rows : 0
Written rows  : 0
